# 4.2 – Producteur de flux – socket Python 1h – Présentiel

### Écrire un script Python simulant un flux de données taxi en émettant les lignes d'un CSV sur une socket TCP, sans charger le fichier en mémoire.



### 1. Lire le CSV ligne par ligne via un itérateur : with open("taxi.csv","r") as f: for line in f: ...


In [ ]:
with open("yellow_tripdata_2026-01.csv", "r") as f:
    for line in f:  # itérateur
        print(line)

### 2. Créer une socket TCP : socket.AF_INET, SOCK_STREAM, bind(("localhost", 9999)), listen, accept


In [ ]:
import socket

# 1. Créer le socket
server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

# 2. Évite l'erreur "Address already in use" si on relance le script
server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

# 3. Attacher le socket au port 9999
server.bind(("localhost", 9999))

# 4. Mettre en écoute (max 1 client en attente)
server.listen(1)

print("En attente d'un client sur le port 9999...")

# 5. Accepter la connexion du client
client, address = server.accept()

print(f"Client connecté depuis {address}")

# teste
# Terminal 1 — lancer le serveur
#python3 producer.py

# Terminal 2 — se connecter
#nc localhost 9999

En attente d'un client sur le port 9999...
Client connecté depuis ('127.0.0.1', 40128)


### 3. Émettre chaque ligne encodée à 100 lignes/s : client.send((line+"\n").encode()); time.sleep(1/100)

In [5]:
import time

server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
server.bind(("localhost", 9999))
server.listen(1)

print("En attente d'un client sur le port 9999...")
client, address = server.accept()
print(f"Client connecté depuis {address}")

with open("yellow_tripdata_2026-01.csv", "r") as f:
    for line in f:
        client.send((line + "\n").encode())  # envoie la ligne encodée
        time.sleep(1/100)                    # pause 10ms = 100 lignes/s

En attente d'un client sur le port 9999...
Client connecté depuis ('127.0.0.1', 43692)


BrokenPipeError: [Errno 32] Broken pipe

### 4. Tester la réception avec nc localhost 9999 dans un autre terminal

SyntaxError: invalid character '’' (U+2019) (2467532992.py, line 1)

### 5. Vérifier que la RAM reste stable quel que soit le volume du fichier (htop)